# 03 - Naming Convention Enforcer

Validates schema, table, and column names against organization naming standards.

**Rules**: lowercase, starts with letter, underscores only, 3-128 chars.


In [0]:
# Databricks notebook source
import sys as _sys
_nb = (dbutils.notebook.entry_point.getDbutils().notebook()
       .getContext().notebookPath().get())
_sys.path.insert(0, '/Workspace' + '/'.join(_nb.split('/')[:-2]) + '/src')
from lib.common import (
    require_widget, uc_list_tables, uc_list_schemas,
    tables_to_spark, build_exempt_schemas,
    load_exemptions, is_exempt,
)
from lib.policy import load_policy, naming_patterns as _policy_naming, anti_patterns as _policy_anti
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("control_schema", "uc_hygiene")
dbutils.widgets.text("target_catalogs", "")
dbutils.widgets.text("table_name_pattern", "^[a-z][a-z0-9_]*$")
dbutils.widgets.text("schema_name_pattern", "^[a-z][a-z0-9_]*$")


catalog        = require_widget(dbutils, "catalog")
control_schema = require_widget(dbutils, "control_schema")
target_catalogs = [
    c.strip() for c in require_widget(dbutils, "target_catalogs").split(",") if c.strip()
]
_widget_table_pattern  = dbutils.widgets.get("table_name_pattern").strip()
_widget_schema_pattern = dbutils.widgets.get("schema_name_pattern").strip()
_policy = load_policy(spark, catalog, control_schema)
table_pattern, schema_pattern = _policy_naming(_policy)
if _widget_table_pattern:  table_pattern  = _widget_table_pattern
if _widget_schema_pattern: schema_pattern = _widget_schema_pattern
control_fqn    = f"{catalog}.{control_schema}"

print(f"Control schema:  {control_fqn}")
print(f"Target catalogs: {target_catalogs}")
print(f"Table pattern:   {table_pattern}")
print(f"Schema pattern:  {schema_pattern}")

In [0]:
# Function definitions moved to src/lib/common.py — imported in widget cell above.
from databricks.sdk import WorkspaceClient

_sdk = WorkspaceClient()


print("✅ lib.common loaded; SDK client ready.")


In [0]:
import re
from datetime import date
import uuid
from pyspark.sql.functions import udf, col, lit, concat_ws, when
from pyspark.sql.types import StringType, BooleanType

scan_id = str(uuid.uuid4())
scan_date = date.today()

# UDF to check naming compliance
@udf(BooleanType())
def is_valid_name(name, pattern):
    if name is None:
        return False
    return bool(re.match(pattern, name))

# Anti-patterns loaded from governance_policy table (editable at runtime via policy.yml + re-run bootstrap)
ANTI_PATTERNS = _policy_anti(_policy)

@udf(StringType())
def check_anti_patterns(name):
    if name is None:
        return None
    findings = []
    for pattern, msg in ANTI_PATTERNS:
        if re.match(pattern, name):
            findings.append(msg)
    return "; ".join(findings) if findings else None

print(f"Anti-patterns to check: {len(ANTI_PATTERNS)}")

In [0]:
# Schema naming check via UC SDK
from pyspark.sql import Row
from pyspark.sql.functions import col, lit, udf
from pyspark.sql.types import BooleanType

_schema_rows = uc_list_schemas(_sdk, target_catalogs, control_schema)
schemas_df   = spark.createDataFrame(
    [Row(catalog_name=r["catalog_name"], schema_name=r["schema_name"]) for r in _schema_rows]
)
schema_violations = schemas_df.withColumn(
    "is_valid", is_valid_name(col("schema_name"), lit(schema_pattern))
).filter(~col("is_valid"))

schema_violation_count = schema_violations.count()
print(f"Schema naming violations: {schema_violation_count}")
if schema_violation_count > 0:
    schema_violations.show(truncate=False)


In [0]:
# Table naming check via UC SDK
_table_rows  = uc_list_tables(_sdk, target_catalogs, control_schema)
_exemptions  = load_exemptions(spark, catalog, control_schema)
_table_rows  = [r for r in _table_rows if not is_exempt(r["catalog_name"], r["schema_name"], r["table_name"], _exemptions)]
tables_df    = tables_to_spark(spark, _table_rows)
table_violations = tables_df.withColumn(
    "is_valid",     is_valid_name(col("table_name"), lit(table_pattern))
).withColumn(
    "anti_pattern", check_anti_patterns(col("table_name"))
).filter((~col("is_valid")) | (col("anti_pattern").isNotNull()))

table_violation_count = table_violations.count()
print(f"Table naming violations: {table_violation_count}")
if table_violation_count > 0:
    table_violations.show(20, truncate=False)


In [0]:
# Column naming check — single SQL query per catalog (columns are metadata, SQL is fine here)
_col_parts = [
    f"SELECT table_catalog, table_schema, table_name, column_name "
    f"FROM {tc}.information_schema.columns "
    f"WHERE table_schema NOT IN ('information_schema','__databricks_internal','uc_hygiene','uc_hygiene_dev','{control_schema}')"
    for tc in target_catalogs
]
columns = spark.sql(" UNION ALL ".join(_col_parts))
col_pattern = "^[a-z][a-z0-9_]*$"
column_violations = columns.withColumn(
    "is_valid", is_valid_name(col("column_name"), lit(col_pattern))
).filter(~col("is_valid"))

col_violation_count = column_violations.count()
print(f"Column naming violations: {col_violation_count}")
if col_violation_count > 0:
    column_violations.show(20, truncate=False)


In [0]:
# Write all findings to control table
# Schema violations
if schema_violation_count > 0:
    schema_violations.createOrReplaceTempView("schema_violations")
    spark.sql(f"""
    INSERT INTO {catalog}.{control_schema}.scan_results
    SELECT
      '{scan_id}', CURRENT_DATE(), 'naming', 'schema',
      sv.catalog_name, sv.schema_name, NULL, NULL,
      'schema_naming_violation', 'warning',
      CONCAT('Schema name does not match pattern: ', '{schema_pattern}'),
      'Rename schema to conform to naming standards',
      NULL, NULL, NULL
    FROM schema_violations sv
    """)

# Table violations
if table_violation_count > 0:
    table_violations.createOrReplaceTempView("table_violations")
    spark.sql(f"""
    INSERT INTO {catalog}.{control_schema}.scan_results
    SELECT
      '{scan_id}', CURRENT_DATE(), 'naming', 'table',
      tv.table_catalog, tv.table_schema, tv.table_name, NULL,
      CASE WHEN tv.anti_pattern IS NOT NULL THEN 'table_anti_pattern' ELSE 'table_naming_violation' END,
      CASE WHEN tv.anti_pattern IS NOT NULL THEN 'info' ELSE 'warning' END,
      COALESCE(tv.anti_pattern, CONCAT('Table name does not match pattern: ', '{table_pattern}')),
      'Rename or restructure table to conform to standards',
      NULL, NULL, NULL
    FROM table_violations tv
    """)

# Column violations (write top 100 to avoid flooding)
if col_violation_count > 0:
    column_violations.limit(100).createOrReplaceTempView("col_violations")
    spark.sql(f"""
    INSERT INTO {catalog}.{control_schema}.scan_results
    SELECT
      '{scan_id}', CURRENT_DATE(), 'naming', 'table',
      cv.table_catalog, cv.table_schema, cv.table_name, cv.column_name,
      'column_naming_violation', 'info',
      'Column name contains uppercase or special characters',
      'Rename column to lowercase with underscores',
      NULL, NULL, NULL
    FROM col_violations cv
    """)

total = schema_violation_count + table_violation_count + col_violation_count
print(f"""
========================================
  NAMING CONVENTION SCAN COMPLETE
========================================
  Scan ID:             {scan_id}
  Schema violations:   {schema_violation_count}
  Table violations:    {table_violation_count}
  Column violations:   {col_violation_count}
  Total findings:      {total}
========================================
""")

In [0]:
# ── Summary & observability ──────────────────────────────────────────────────
total_violations = schema_violation_count + table_violation_count + col_violation_count
print(f"""
{'='*52}
  NAMING CONVENTION SCAN COMPLETE
{'='*52}
  Scan ID:            {scan_id}
  Schema violations:  {schema_violation_count}
  Table violations:   {table_violation_count}
  Column violations:  {col_violation_count}
  Total violations:   {total_violations}
  Results → {catalog}.{control_schema}.scan_results
{'='*52}
""")

total_tables = len(_table_rows)
try:
    spark.sql(f"""
    INSERT INTO {catalog}.{control_schema}.job_run_history VALUES (
      CURRENT_DATE(),
      'uc_hygiene_daily_governance',
      'p2_naming_conventions',
      'p2_detection',
      'success',
      {total_tables},
      {total_violations},
      {total_violations},
      NULL,
      'schema={schema_violation_count} table={table_violation_count} col={col_violation_count}',
      CURRENT_TIMESTAMP()
    )
    """)
except Exception as _e:
    print(f"Warning: could not write to job_run_history: {_e}")